# vLLM Tutorial 3: Automatic Prefix Caching (APC)

## Overview

This notebook covers:

- What is Automatic Prefix Caching (APC)

- How APC works internally

- When to use APC

- Implementation and configuration

- Real-world use cases

- Performance benchmarks

- Best practices

---

## What is Automatic Prefix Caching?

**Automatic Prefix Caching (APC)** is a feature in vLLM that automatically detects and reuses the KV cache from common prompt prefixes.

### The Problem Without APC

Imagine you're building a chatbot that starts every conversation with the same system prompt:

In [ ]:
User 1: [System: You are a helpful assistant...] + "What is AI?"
        ↓ Process entire prompt (1000 tokens)

User 2: [System: You are a helpful assistant...] + "How's the weather?"
        ↓ Process entire prompt again (1000 tokens) ⚠️ Wasteful!

User 3: [System: You are a helpful assistant...] + "Tell me a joke"
        ↓ Process entire prompt again (1000 tokens) ⚠️ Wasteful!

**Problem**: The system prompt is recomputed for every request, wasting GPU cycles and memory.

### The Solution: Automatic Prefix Caching

In [ ]:
User 1: [System: You are a helpful assistant...] + "What is AI?"
        ↓ Process and CACHE prefix (1000 tokens)

User 2: [System: You are a helpful assistant...] + "How's the weather?"
        ↓ REUSE cached prefix + process new part (20 tokens) ✅ Fast!

User 3: [System: You are a helpful assistant...] + "Tell me a joke"
        ↓ REUSE cached prefix + process new part (15 tokens) ✅ Fast!

**Benefits**:

- ⚡ **Faster response times** (skip recomputing prefix)

- 💰 **Lower costs** (less GPU computation)

- 📈 **Higher throughput** (serve more requests)

- 🎯 **Automatic** (no code changes needed)

---

## How APC Works Internally

### 1. Prefix Detection

vLLM automatically detects common prefixes across requests:

In [ ]:
# Request 1
prompt1 = "You are a helpful AI. The capital of France is"
#          [─────────Prefix─────────] [──New Part──]

# Request 2  
prompt2 = "You are a helpful AI. What is 2+2?"
#          [─────────Prefix─────────] [──New Part──]
#          ✅ Same prefix → Cache hit!

### 2. KV Cache Storage

When a prefix is processed:

1. **Compute KV cache** for the prefix tokens

2. **Store in cache** with prefix hash as key

3. **Mark as reusable** for future requests

In [ ]:
Cache Structure:
┌─────────────────────────────────────┐
│ Prefix: "You are a helpful AI."     │
│ Hash: 0x7a3f2b1c                    │
│ KV Cache: [Layer1, Layer2, ...]    │
│ Size: 512 tokens                    │
│ Last Used: 2s ago                   │
└─────────────────────────────────────┘

### 3. Cache Reuse

When a new request arrives:

1. **Hash the prompt** to check for prefix match

2. **Find longest matching prefix** in cache

3. **Reuse cached KV** for prefix tokens

4. **Compute only new tokens** after the prefix

In [ ]:
# Pseudo-code
def process_request(prompt):
    prefix_hash = hash(prompt[:N])  # Hash first N tokens

    if prefix_hash in cache:
        kv_cache = cache[prefix_hash]  # Reuse!
        remaining_tokens = prompt[N:]   # Only process this
    else:
        kv_cache = compute_kv(prompt)  # Full computation
        cache[prefix_hash] = kv_cache  # Store for future

### 4. Cache Eviction

When cache is full, vLLM uses **LRU (Least Recently Used)** eviction:

- Tracks when each cached prefix was last used

- Evicts oldest unused prefixes

- Automatically manages memory

---

## Enabling Automatic Prefix Caching

APC is **enabled by default** in vLLM, but let's see how to configure it.

### Method 1: Python API (LLM Class)

In [ ]:
from vllm import LLM, SamplingParams

# APC is enabled by default
llm = LLM(
    model="facebook/opt-125m",
    # enable_prefix_caching=True,  # Default: True (no need to specify)
    trust_remote_code=True
)

print("LLM initialized with Automatic Prefix Caching enabled")

### Method 2: API Server (Command Line)

In [ ]:
# Start API server with APC (enabled by default)
python -m vllm.entrypoints.openai.api_server \
    --model meta-llama/Llama-2-7b-hf \
    --enable-prefix-caching \
    --port 8000

# Disable APC (if needed)
python -m vllm.entrypoints.openai.api_server \
    --model meta-llama/Llama-2-7b-hf \
    --disable-prefix-caching \
    --port 8000

---

## Demonstrating APC Benefits

Let's measure the performance improvement with APC.

In [ ]:
import time
from vllm import LLM, SamplingParams

# Initialize model
llm = LLM(model="facebook/opt-125m", trust_remote_code=True)

# Common system prompt (this will be cached)
system_prompt = """You are a helpful AI assistant specialized in answering questions 
about technology, science, and programming. You provide clear, accurate, and 
beginner-friendly explanations. Always be respectful and professional.

User question: """

# Different user questions
questions = [
    "What is machine learning?",
    "How does the internet work?",
    "Explain quantum computing.",
    "What is blockchain?",
    "How do neural networks work?"
]

# Create full prompts with same prefix
prompts = [system_prompt + q for q in questions]

sampling_params = SamplingParams(temperature=0.7, max_tokens=50)

print("Testing Automatic Prefix Caching...\n")
print(f"System prompt length: {len(system_prompt.split())} words")
print(f"Number of requests: {len(prompts)}")
print("=" * 80)

# Process all requests
start_time = time.time()
outputs = llm.generate(prompts, sampling_params)
total_time = time.time() - start_time

print(f"\nTotal time: {total_time:.2f} seconds")
print(f"Average time per request: {total_time / len(prompts):.2f} seconds")
print("\nNote: First request is slower (processes full prompt)")
print("Subsequent requests are faster (reuse cached prefix)")

### Visualizing Cache Hits

Let's process requests one by one to see the speedup.

In [ ]:
import time

# Reinitialize to clear cache
llm = LLM(model="facebook/opt-125m", trust_remote_code=True)

system_prompt = "You are a helpful assistant. Answer concisely. Question: "
questions = [
    "What is AI?",
    "What is ML?",
    "What is DL?",
]

sampling_params = SamplingParams(temperature=0.7, max_tokens=30)

times = []
for i, question in enumerate(questions, 1):
    prompt = system_prompt + question

    start = time.time()
    output = llm.generate([prompt], sampling_params)[0]
    elapsed = time.time() - start
    times.append(elapsed)

    cache_status = "❄️  CACHE MISS" if i == 1 else "✅ CACHE HIT"
    print(f"Request {i}: {elapsed:.3f}s {cache_status}")
    print(f"  Q: {question}")
    print(f"  A: {output.outputs[0].text.strip()[:60]}...")
    print()

# Show speedup
if len(times) > 1:
    speedup = times[0] / times[1]
    print(f"Speedup from caching: {speedup:.2f}x faster")
    print(f"Time saved: {(times[0] - times[1]):.3f}s per request")

---

## Real-World Use Cases for APC

APC provides maximum benefit in these scenarios:

### Use Case 1: Chatbots with System Prompts

**Scenario**: Every conversation starts with the same system prompt.

**Without APC**:

- Each request processes full system prompt (1000+ tokens)

- Wastes GPU time and memory

**With APC**:

- System prompt cached after first request

- 5-10x speedup for subsequent requests

In [ ]:
# Chatbot example with system prompt
SYSTEM_PROMPT = """You are a customer service AI for TechCorp. 
Guidelines:
- Be polite and professional
- Provide accurate information
- If unsure, offer to escalate to human agent
- Keep responses concise

Customer: """

# Simulate multiple customer queries
customer_queries = [
    "How do I reset my password?",
    "What are your business hours?",
    "I need to cancel my subscription",
]

sampling_params = SamplingParams(temperature=0.3, max_tokens=80)

for query in customer_queries:
    prompt = SYSTEM_PROMPT + query
    output = llm.generate([prompt], sampling_params)[0]
    print(f"Customer: {query}")
    print(f"AI: {output.outputs[0].text.strip()}")
    print("-" * 80)

### Use Case 2: Few-Shot Learning

**Scenario**: Provide examples (few-shot) in every prompt.

Few-shot prompts contain examples that teach the model a task:

In [ ]:
# Few-shot sentiment analysis
FEW_SHOT_PREFIX = """Classify the sentiment as Positive, Negative, or Neutral.

Examples:
Text: "I love this product!" → Sentiment: Positive
Text: "This is terrible." → Sentiment: Negative
Text: "It's okay, nothing special." → Sentiment: Neutral
Text: "Best purchase ever!" → Sentiment: Positive
Text: "Waste of money." → Sentiment: Negative

Now classify this:
Text: """

# New texts to classify
texts = [
    "Amazing quality and fast shipping!",
    "Disappointed with the quality.",
    "It works as expected.",
]

sampling_params = SamplingParams(temperature=0.1, max_tokens=10)

for text in texts:
    prompt = FEW_SHOT_PREFIX + text + '" → Sentiment:'
    output = llm.generate([prompt], sampling_params)[0]
    sentiment = output.outputs[0].text.strip()
    print(f"Text: {text}")
    print(f"Sentiment: {sentiment}")
    print()

### Use Case 3: Document Analysis with Context

**Scenario**: Analyze a long document with multiple questions.

**Benefit**: Document is cached, only questions change.

In [ ]:
# Long document (this will be cached)
DOCUMENT = """The Python programming language was created by Guido van Rossum 
and first released in 1991. Python is known for its simple, readable syntax 
that emphasizes code readability. It supports multiple programming paradigms 
including procedural, object-oriented, and functional programming. Python has 
a comprehensive standard library and a vast ecosystem of third-party packages 
available through PyPI (Python Package Index). It's widely used in web 
development, data science, machine learning, automation, and scientific computing.

Question: """

# Multiple questions about the same document
questions = [
    "Who created Python?",
    "When was Python first released?",
    "What programming paradigms does Python support?",
    "What is PyPI?",
]

sampling_params = SamplingParams(temperature=0.2, max_tokens=30)

print("Answering multiple questions about the same document...\n")
for i, question in enumerate(questions, 1):
    prompt = DOCUMENT + question
    output = llm.generate([prompt], sampling_params)[0]

    cache_status = "(First request - caching)" if i == 1 else "(Using cache)"
    print(f"Q{i}: {question} {cache_status}")
    print(f"A: {output.outputs[0].text.strip()}")
    print()

### Use Case 4: Code Generation with Templates

**Scenario**: Generate code with consistent templates/imports.

In [ ]:
# Code template (will be cached)
CODE_TEMPLATE = """You are a Python code generator. Generate clean, well-documented code.

Standard imports:
import os
import sys
from typing import List, Dict, Optional

Task: """

tasks = [
    "Create a function to check if a number is prime",
    "Create a function to reverse a string",
    "Create a function to find the maximum in a list",
]

sampling_params = SamplingParams(temperature=0.4, max_tokens=100)

for task in tasks:
    prompt = CODE_TEMPLATE + task
    output = llm.generate([prompt], sampling_params)[0]
    print(f"Task: {task}")
    print("Generated code:")
    print(output.outputs[0].text)
    print("=" * 80)

---

## Performance Benchmarks

Let's measure the actual performance improvement with different prefix sizes.

In [ ]:
import time
import numpy as np

def benchmark_prefix_caching(prefix_size_words, num_requests=5):
    """Benchmark APC with different prefix sizes"""

    # Create prefix of specified size
    words = ["word"] * prefix_size_words
    prefix = " ".join(words) + " Question: "

    # Create requests with same prefix but different suffixes
    prompts = [prefix + f"What is example {i}?" for i in range(num_requests)]

    sampling_params = SamplingParams(temperature=0.7, max_tokens=20)

    # Measure time for each request
    times = []
    for i, prompt in enumerate(prompts):
        start = time.time()
        _ = llm.generate([prompt], sampling_params)
        elapsed = time.time() - start
        times.append(elapsed)

    # Calculate statistics
    first_request_time = times[0]
    avg_cached_time = np.mean(times[1:]) if len(times) > 1 else 0
    speedup = first_request_time / avg_cached_time if avg_cached_time > 0 else 1

    return {
        'prefix_size': prefix_size_words,
        'first_time': first_request_time,
        'cached_time': avg_cached_time,
        'speedup': speedup
    }

# Test different prefix sizes
print("Benchmarking Automatic Prefix Caching...\n")
print(f"{'Prefix Size':<15} {'First Request':<15} {'Cached Request':<15} {'Speedup':<10}")
print("=" * 60)

for prefix_size in [50, 100, 200, 500]:
    result = benchmark_prefix_caching(prefix_size, num_requests=3)
    print(f"{result['prefix_size']:<15} "
          f"{result['first_time']:.3f}s{' ':<10} "
          f"{result['cached_time']:.3f}s{' ':<10} "
          f"{result['speedup']:.2f}x")

print("\nKey Insight: Larger prefixes = Greater speedup from caching")

---

## Best Practices for APC

### 1. Structure Prompts for Maximum Caching

In [ ]:
✅ **Good**: Common prefix at the beginning
prompt = SYSTEM_PROMPT + user_input  # System prompt cached

In [ ]:
❌ **Bad**: Variable parts at the beginning
prompt = user_input + SYSTEM_PROMPT  # Can't cache effectively

### 2. Use Consistent Formatting

In [ ]:
✅ **Good**: Consistent formatting
prompts = [
    f"{TEMPLATE}Question: {q}" for q in questions
]
# All prompts share same prefix

In [ ]:
❌ **Bad**: Inconsistent formatting
prompts = [
    f"{TEMPLATE}Question: {q}",
    f"{TEMPLATE}Q: {q2}",  # Different format
]
# Different prefixes, can't cache

### 3. Optimal Prefix Length

- **Too short** (<50 tokens): Minimal benefit

- **Sweet spot** (100-1000 tokens): Maximum speedup

- **Very long** (>2000 tokens): Diminishing returns

### 4. Monitor Cache Hit Rate

Track how often your prefixes are being reused:

In [ ]:
# In production, monitor:
# - Cache hit rate (higher is better)
# - Average prefix length
# - Time savings from caching

### 5. Consider Cache Size

vLLM automatically manages cache size, but be aware:

- Cache competes with KV cache for GPU memory

- LRU eviction removes old prefixes

- More diverse prefixes = lower hit rate

---

## Advanced: Controlling Cache Behavior

### Disabling APC for Specific Requests

Sometimes you may want to disable caching for certain requests:

In [ ]:
# Initialize with APC disabled
llm_no_cache = LLM(
    model="facebook/opt-125m",
    enable_prefix_caching=False,  # Disable APC
    trust_remote_code=True
)

print("LLM initialized WITHOUT prefix caching")
print("Use this when:")
print("  - Every request has unique prompts")
print("  - You want deterministic performance")
print("  - Debugging/benchmarking")

---

## Common Pitfalls and Solutions

### Pitfall 1: Low Cache Hit Rate

**Problem**: Prefixes are too diverse

In [ ]:
**Solution**: Standardize prompt templates
# Bad: Different templates
prompts = [
    f"Answer this: {q1}",
    f"Please respond: {q2}",
    f"Question: {q3}"
]

# Good: Same template
template = "Answer this question: "
prompts = [template + q for q in questions]

### Pitfall 2: Variable Injection in Prefix

**Problem**: User-specific data in prefix

In [ ]:
**Solution**: Move variables to suffix
# Bad: User ID in prefix (no caching)
prompt = f"User {user_id}: {SYSTEM_PROMPT} {question}"

# Good: User ID in suffix (caching works)
prompt = f"{SYSTEM_PROMPT} User {user_id} asks: {question}"

### Pitfall 3: Timestamps in Prompts

**Problem**: Including current time prevents caching

In [ ]:
**Solution**: Remove or move timestamps
# Bad: Timestamp changes every second
prompt = f"[{datetime.now()}] {SYSTEM} {question}"

# Good: No timestamp, or at end
prompt = f"{SYSTEM} {question} (Time: {datetime.now()})"

---

## Practical Example: Building a Cached Q&A System

Let's build a complete Q&A system optimized for APC.

In [ ]:
class CachedQASystem:
    """Q&A system optimized for Automatic Prefix Caching"""

    def __init__(self, model_name="facebook/opt-125m", context=""):
        self.llm = LLM(model=model_name, trust_remote_code=True)

        # This prefix will be cached after first use
        self.prefix = f"""You are a helpful Q&A assistant.

Context:
{context}

Instructions:
- Answer based on the context above
- Be concise and accurate
- If unsure, say "I don't know"

Question: """

        self.sampling_params = SamplingParams(
            temperature=0.3,
            max_tokens=100
        )

    def ask(self, question):
        """Ask a question (benefits from prefix caching)"""
        prompt = self.prefix + question
        output = self.llm.generate([prompt], self.sampling_params)[0]
        return output.outputs[0].text.strip()

    def ask_batch(self, questions):
        """Ask multiple questions efficiently"""
        prompts = [self.prefix + q for q in questions]
        outputs = self.llm.generate(prompts, self.sampling_params)
        return [out.outputs[0].text.strip() for out in outputs]

# Example usage
context = """Python is a high-level programming language created by 
Guido van Rossum in 1991. It emphasizes code readability and simplicity."""

qa_system = CachedQASystem(context=context)

# Ask multiple questions (prefix cached after first)
questions = [
    "Who created Python?",
    "When was Python created?",
    "What does Python emphasize?"
]

print("Cached Q&A System Demo\n")
for q in questions:
    answer = qa_system.ask(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

---

## Comparing: With vs Without APC

Let's compare performance with and without APC.

In [ ]:
import time

# Setup
system_prompt = "You are an expert assistant. " * 50  # Long prefix
questions = [f"Question {i}?" for i in range(5)]
prompts = [system_prompt + q for q in questions]
sampling_params = SamplingParams(temperature=0.7, max_tokens=20)

# Test WITH APC
llm_with_apc = LLM(model="facebook/opt-125m", enable_prefix_caching=True)
start = time.time()
outputs = llm_with_apc.generate(prompts, sampling_params)
time_with_apc = time.time() - start

# Test WITHOUT APC
llm_no_apc = LLM(model="facebook/opt-125m", enable_prefix_caching=False)
start = time.time()
outputs = llm_no_apc.generate(prompts, sampling_params)
time_no_apc = time.time() - start

# Results
print("Performance Comparison")
print("=" * 50)
print(f"With APC:    {time_with_apc:.3f}s")
print(f"Without APC: {time_no_apc:.3f}s")
print(f"Speedup:     {time_no_apc / time_with_apc:.2f}x")
print(f"Time saved:  {time_no_apc - time_with_apc:.3f}s")

---

## Summary

In this notebook, you learned:

- ✅ What Automatic Prefix Caching (APC) is and why it matters

- ✅ How APC works internally (prefix detection, KV cache reuse, LRU eviction)

- ✅ How to enable and configure APC

- ✅ Real-world use cases (chatbots, few-shot, document analysis, code generation)

- ✅ Performance benchmarks showing 2-10x speedup

- ✅ Best practices for maximizing cache hits

- ✅ Common pitfalls and how to avoid them

### Key Takeaways

1. **APC is automatic** - No code changes needed, works out of the box

2. **Structure matters** - Put common parts at the beginning of prompts

3. **Bigger prefixes = Bigger speedup** - 100-1000 tokens is optimal

4. **Consistency is key** - Use standardized prompt templates

5. **Perfect for production** - Reduces costs and improves user experience

### Next Steps

Continue to the next notebook to learn about:

- Continuous batching implementation

- Advanced features (quantization, multi-GPU)

- Production optimization techniques